# NB01: Schema Discovery and Evidence Channel Characterization

**Purpose**: Before building the mapping pipeline, discover exactly what data is available in BERDL for linking UniProt proteins to ModelSEED reactions.

**Key questions**:
1. What are the distinct `status` values in `reaction`? Which ones are mass-balanced?
2. What cross-reference types (`db` values) exist in `uniprot_identifier`?
3. What does `u_seaver__msd_biochemistry` contain that the standard DB lacks?
4. What format are KEGG, BioCyc, Reactome entries in `uniprot_identifier`?
5. What does `reaction.abbreviation` actually contain (KEGG R-numbers? EC patterns?)?
6. What are the schemas of `curatedgene`, `interproscan_pathways`, `seedannotation`?

**Requires**: BERDL JupyterHub (Spark session)

In [ ]:
import os, sys
import pandas as pd

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
except ImportError:
    from get_spark_session import get_spark_session

spark = get_spark_session()

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

spark.sql("SET spark.sql.autoBroadcastJoinThreshold = -1")
print('Spark session ready. Auto-broadcast disabled.')

## 1. ModelSEED Biochemistry: Reaction Status Values

The `reaction.status` column flags mass-balanced reactions. Enumerate all values.

In [ ]:
reaction_status = spark.sql("""
    SELECT status, COUNT(*) as n_reactions
    FROM kbase_msd_biochemistry.reaction
    GROUP BY status
    ORDER BY n_reactions DESC
""").toPandas()

print('=== Reaction Status Distribution ===')
print(reaction_status.to_string(index=False))
print(f'\nTotal reactions: {reaction_status.n_reactions.sum():,}')
reaction_status.to_csv(f'{DATA_DIR}/reaction_status_counts.csv', index=False)

In [ ]:
reaction_schema = spark.sql("DESCRIBE EXTENDED kbase_msd_biochemistry.reaction").toPandas()
print('=== kbase_msd_biochemistry.reaction schema ===')
print(reaction_schema[['col_name', 'data_type']].to_string(index=False))

## 2. Reaction Abbreviation Patterns

Check what `abbreviation` actually contains — KEGG R-numbers, EC-like patterns, MetaCyc IDs?

In [ ]:
abbrev_stats = spark.sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN abbreviation IS NOT NULL AND abbreviation != '' THEN 1 ELSE 0 END) as has_abbreviation,
        SUM(CASE WHEN abbreviation RLIKE '^R[0-9]{5}' THEN 1 ELSE 0 END) as kegg_r_number,
        SUM(CASE WHEN abbreviation RLIKE '[0-9]+\\.[0-9]+\\.[0-9]+\\.[0-9]+' THEN 1 ELSE 0 END) as has_ec_pattern,
        SUM(CASE WHEN abbreviation LIKE '%-RXN%' THEN 1 ELSE 0 END) as metacyc_rxn_pattern
    FROM kbase_msd_biochemistry.reaction
""").toPandas()

print('=== Abbreviation Pattern Analysis ===')
for col in abbrev_stats.columns:
    print(f'  {col}: {abbrev_stats[col].iloc[0]:,}')

In [ ]:
abbrev_samples = spark.sql("""
    SELECT abbreviation, COUNT(*) as n
    FROM kbase_msd_biochemistry.reaction
    WHERE abbreviation IS NOT NULL AND abbreviation != ''
    GROUP BY abbreviation
    ORDER BY n DESC
    LIMIT 30
""").toPandas()

print('=== Top 30 Most Common Abbreviations ===')
print(abbrev_samples.to_string(index=False))

In [ ]:
print('=== Sample abbreviation values (diverse patterns) ===')
samples = spark.sql("""
    (
        SELECT id, name, abbreviation, status, 'kegg_r' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation RLIKE '^R[0-9]{5}'
        LIMIT 5
    )
    UNION ALL
    (
        SELECT id, name, abbreviation, status, 'ec_like' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation RLIKE '[0-9]+\\.[0-9]+\\.[0-9]+\\.[0-9]+'
            AND NOT abbreviation RLIKE '^R[0-9]{5}'
        LIMIT 5
    )
    UNION ALL
    (
        SELECT id, name, abbreviation, status, 'metacyc' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation LIKE '%-RXN%'
        LIMIT 5
    )
    UNION ALL
    (
        SELECT id, name, abbreviation, status, 'other' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation IS NOT NULL
            AND abbreviation != ''
            AND NOT abbreviation RLIKE '^R[0-9]{5}'
            AND NOT abbreviation RLIKE '[0-9]+\\.[0-9]+\\.[0-9]+\\.[0-9]+'
            AND abbreviation NOT LIKE '%-RXN%'
        LIMIT 5
    )
""").toPandas()

print(samples.to_string(index=False))

## 3. User's Augmented Biochemistry Database

Compare `u_seaver__msd_biochemistry` (6 tables) vs `kbase_msd_biochemistry` (5 tables).

In [ ]:
std_tables = spark.sql("SHOW TABLES IN kbase_msd_biochemistry").toPandas()
print('=== kbase_msd_biochemistry tables ===')
print(std_tables.to_string(index=False))

print()

try:
    user_tables = spark.sql("SHOW TABLES IN u_seaver__msd_biochemistry").toPandas()
    print('=== u_seaver__msd_biochemistry tables ===')
    print(user_tables.to_string(index=False))

    std_set = set(std_tables['tableName'])
    user_set = set(user_tables['tableName'])
    extra = user_set - std_set
    if extra:
        print(f'\nExtra tables in user DB: {extra}')
        for t in extra:
            print(f'\n--- Schema for {t} ---')
            schema = spark.sql(f"DESCRIBE EXTENDED u_seaver__msd_biochemistry.{t}").toPandas()
            print(schema[['col_name', 'data_type']].to_string(index=False))
            print(f'\n--- Sample rows from {t} ---')
            sample = spark.sql(f"SELECT * FROM u_seaver__msd_biochemistry.{t} LIMIT 5").toPandas()
            print(sample.to_string(index=False))
    else:
        print('\nNo extra tables found.')
except Exception as e:
    print(f'Could not access u_seaver__msd_biochemistry: {e}')

In [ ]:
try:
    user_rxn_schema = spark.sql("DESCRIBE EXTENDED u_seaver__msd_biochemistry.reaction").toPandas()
    std_rxn_schema = spark.sql("DESCRIBE EXTENDED kbase_msd_biochemistry.reaction").toPandas()

    user_cols = set(user_rxn_schema['col_name'])
    std_cols = set(std_rxn_schema['col_name'])
    extra_cols = user_cols - std_cols
    if extra_cols:
        print(f'Extra columns in user reaction table: {extra_cols}')
        for col in extra_cols:
            sample = spark.sql(f"""
                SELECT `{col}`, COUNT(*) as n
                FROM u_seaver__msd_biochemistry.reaction
                WHERE `{col}` IS NOT NULL
                GROUP BY `{col}`
                ORDER BY n DESC
                LIMIT 20
            """).toPandas()
            print(f'\nTop values for {col}:')
            print(sample.to_string(index=False))
    else:
        print('Same columns in user and standard reaction tables.')
except Exception as e:
    print(f'Could not compare reaction schemas: {e}')

## 4. UniProt Databases: Table Inventory and Cross-Reference Types

Enumerate all tables in both UniProt databases, then characterize `uniprot_identifier`.

In [ ]:
for db_name in ['refdata_uniprot', 'kbase_uniprot_kb']:
    try:
        tables = spark.sql(f"SHOW TABLES IN {db_name}").toPandas()
        print(f'=== {db_name} ({len(tables)} tables) ===')
        print(tables.to_string(index=False))
        print()

        for _, row in tables.iterrows():
            tbl = row['tableName']
            try:
                schema = spark.sql(f"DESCRIBE {db_name}.{tbl}").toPandas()
                cols = ', '.join(schema['col_name'].tolist())
                cnt = spark.sql(f"SELECT COUNT(*) as n FROM {db_name}.{tbl}").collect()[0]['n']
                print(f'  {tbl} ({cnt:,} rows): {cols}')
            except Exception as e:
                print(f'  {tbl}: ERROR - {e}')
        print()
    except Exception as e:
        print(f'{db_name}: ERROR - {e}\n')

In [ ]:
print('=== uniprot_identifier: distinct db values and counts ===')
print('(This query aggregates 2.5B rows — may take a few minutes)\n')

for db_name in ['refdata_uniprot', 'kbase_uniprot_kb']:
    try:
        db_values = spark.sql(f"""
            SELECT db, COUNT(*) as n_rows, COUNT(DISTINCT uniprot_id) as n_proteins
            FROM {db_name}.uniprot_identifier
            GROUP BY db
            ORDER BY n_rows DESC
        """).toPandas()

        print(f'--- {db_name}.uniprot_identifier ---')
        print(db_values.to_string(index=False))
        print(f'Total rows: {db_values.n_rows.sum():,}')
        print()

        db_values.to_csv(f'{DATA_DIR}/db_value_counts_{db_name}.csv', index=False)
    except Exception as e:
        print(f'{db_name}.uniprot_identifier: ERROR - {e}\n')

In [ ]:
print('=== Sample entries per db type (20 rows each) ===\n')

target_db = 'refdata_uniprot'  # or kbase_uniprot_kb — adjust after cell above

try:
    db_types = spark.sql(f"""
        SELECT DISTINCT db FROM {target_db}.uniprot_identifier
    """).toPandas()['db'].tolist()

    for db_type in sorted(db_types):
        sample = spark.sql(f"""
            SELECT uniprot_id, db, xref
            FROM {target_db}.uniprot_identifier
            WHERE db = '{db_type}'
            LIMIT 10
        """).toPandas()
        print(f'--- db = {db_type} ---')
        print(sample.to_string(index=False))
        print()
except Exception as e:
    print(f'Error sampling db types: {e}')

## 5. UniRef Databases

Check what UniRef clustering data is available for protein family grouping.

In [ ]:
uniref_dbs = spark.sql("SHOW DATABASES").toPandas()
uniref_dbs = uniref_dbs[uniref_dbs['namespace'].str.contains('uniref', case=False, na=False)]
print('=== UniRef databases ===')
print(uniref_dbs.to_string(index=False))

for _, row in uniref_dbs.iterrows():
    db_name = row['namespace']
    tables = spark.sql(f"SHOW TABLES IN {db_name}").toPandas()
    print(f'\n--- {db_name} ({len(tables)} tables) ---')
    for _, trow in tables.iterrows():
        tbl = trow['tableName']
        try:
            schema = spark.sql(f"DESCRIBE {db_name}.{tbl}").toPandas()
            cols = ', '.join(schema['col_name'].tolist())
            print(f'  {tbl}: {cols}')
        except Exception as e:
            print(f'  {tbl}: ERROR - {e}')

## 6. PaperBLAST: curatedgene Schema

Understand what curated gene→reaction data is available.

In [ ]:
pb_tables = spark.sql("SHOW TABLES IN kescience_paperblast").toPandas()
print('=== kescience_paperblast tables ===')
print(pb_tables.to_string(index=False))

for tbl in ['curatedgene', 'seqtoduplicate', 'gene']:
    try:
        schema = spark.sql(f"DESCRIBE kescience_paperblast.{tbl}").toPandas()
        print(f'\n--- {tbl} schema ---')
        print(schema[['col_name', 'data_type']].to_string(index=False))

        cnt = spark.sql(f"SELECT COUNT(*) as n FROM kescience_paperblast.{tbl}").collect()[0]['n']
        print(f'Rows: {cnt:,}')

        print(f'\n--- {tbl} sample (5 rows) ---')
        sample = spark.sql(f"SELECT * FROM kescience_paperblast.{tbl} LIMIT 5").toPandas()
        print(sample.to_string(index=False))
    except Exception as e:
        print(f'{tbl}: ERROR - {e}')
    print()

In [ ]:
print('=== curatedgene: distinct db values ===')
try:
    cg_dbs = spark.sql("""
        SELECT db, COUNT(*) as n
        FROM kescience_paperblast.curatedgene
        GROUP BY db
        ORDER BY n DESC
    """).toPandas()
    print(cg_dbs.to_string(index=False))
except Exception as e:
    print(f'Error: {e}')

## 7. InterPro: protein2ipr and interproscan_pathways

Check what domain→EC→reaction evidence is available.

In [ ]:
ip_tables = spark.sql("SHOW TABLES IN kescience_interpro").toPandas()
print('=== kescience_interpro tables ===')
print(ip_tables.to_string(index=False))

for tbl in ip_tables['tableName'].tolist():
    try:
        schema = spark.sql(f"DESCRIBE kescience_interpro.{tbl}").toPandas()
        print(f'\n--- {tbl} schema ---')
        print(schema[['col_name', 'data_type']].to_string(index=False))

        cnt = spark.sql(f"SELECT COUNT(*) as n FROM kescience_interpro.{tbl}").collect()[0]['n']
        print(f'Rows: {cnt:,}')
    except Exception as e:
        print(f'{tbl}: ERROR - {e}')
    print()

In [ ]:
print('=== InterPro entries with EC associations ===')
try:
    schema = spark.sql("DESCRIBE kescience_interpro.entry").toPandas()
    print('entry schema:', ', '.join(schema['col_name'].tolist()))
    print()

    sample = spark.sql("""
        SELECT *
        FROM kescience_interpro.entry
        LIMIT 10
    """).toPandas()
    print(sample.to_string(index=False))
except Exception as e:
    print(f'Error: {e}')

In [ ]:
print('=== interproscan_pathways: check for KEGG/MetaCyc pathway annotations ===')

for db_name in ['kbase_ke_pangenome']:
    try:
        tables = spark.sql(f"SHOW TABLES IN {db_name}").toPandas()
        ips_tables = tables[tables['tableName'].str.contains('interpro', case=False)]
        if len(ips_tables) > 0:
            print(f'InterPro-related tables in {db_name}:')
            print(ips_tables.to_string(index=False))
            for _, row in ips_tables.iterrows():
                tbl = row['tableName']
                schema = spark.sql(f"DESCRIBE {db_name}.{tbl}").toPandas()
                print(f'\n  {tbl}: {list(schema["col_name"])}')
                sample = spark.sql(f"SELECT * FROM {db_name}.{tbl} LIMIT 5").toPandas()
                print(sample.to_string(index=False))
        else:
            print(f'No InterPro tables in {db_name}')
    except Exception as e:
        print(f'{db_name}: {e}')

## 8. Fitness Browser: SEED Annotations

Check `seedannotation` and `seedclass` for SEED role→reaction links.

In [ ]:
for tbl in ['seedannotation', 'seedclass', 'besthitkegg', 'besthitmetacyc', 'besthitswissprot']:
    try:
        schema = spark.sql(f"DESCRIBE kescience_fitnessbrowser.{tbl}").toPandas()
        cnt = spark.sql(f"SELECT COUNT(*) as n FROM kescience_fitnessbrowser.{tbl}").collect()[0]['n']
        print(f'--- {tbl} ({cnt:,} rows) ---')
        print(f'Columns: {list(schema["col_name"])}')

        sample = spark.sql(f"SELECT * FROM kescience_fitnessbrowser.{tbl} LIMIT 3").toPandas()
        print(sample.to_string(index=False))
        print()
    except Exception as e:
        print(f'{tbl}: ERROR - {e}\n')

## 9. eggNOG and bakta: EC/KEGG/UniRef Annotations

In [ ]:
print('=== eggnog_mapper_annotations: EC/KEGG coverage ===')
ec_coverage = spark.sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN EC IS NOT NULL AND EC != '' AND EC != '-' THEN 1 ELSE 0 END) as has_ec,
        SUM(CASE WHEN KEGG_Reaction IS NOT NULL AND KEGG_Reaction != '' AND KEGG_Reaction != '-' THEN 1 ELSE 0 END) as has_kegg_rxn,
        SUM(CASE WHEN KEGG_ko IS NOT NULL AND KEGG_ko != '' AND KEGG_ko != '-' THEN 1 ELSE 0 END) as has_kegg_ko,
        SUM(CASE WHEN BiGG_Reaction IS NOT NULL AND BiGG_Reaction != '' AND BiGG_Reaction != '-' THEN 1 ELSE 0 END) as has_bigg
    FROM kbase_ke_pangenome.eggnog_mapper_annotations
""").toPandas()

total = ec_coverage['total'].iloc[0]
for col in ec_coverage.columns:
    val = ec_coverage[col].iloc[0]
    pct = val / total * 100 if total > 0 else 0
    print(f'  {col}: {val:,} ({pct:.1f}%)')

In [ ]:
print('=== bakta_annotations: EC/UniRef coverage ===')
bakta_coverage = spark.sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN ec IS NOT NULL AND ec != '' THEN 1 ELSE 0 END) as has_ec
    FROM kbase_ke_pangenome.bakta_annotations
""").toPandas()

total = bakta_coverage['total'].iloc[0]
for col in bakta_coverage.columns:
    val = bakta_coverage[col].iloc[0]
    pct = val / total * 100 if total > 0 else 0
    print(f'  {col}: {val:,} ({pct:.1f}%)')

print('\n=== bakta_db_xrefs: distinct db values ===')
bakta_dbs = spark.sql("""
    SELECT db, COUNT(*) as n
    FROM kbase_ke_pangenome.bakta_db_xrefs
    GROUP BY db
    ORDER BY n DESC
""").toPandas()
print(bakta_dbs.to_string(index=False))

## 10. Cache Small Reference Tables

Download the reaction, molecule, and reagent tables for local use in downstream notebooks.

In [ ]:
reactions = spark.sql("SELECT * FROM kbase_msd_biochemistry.reaction").toPandas()
reactions.to_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', index=False)
print(f'Cached {len(reactions):,} reactions to data/reactions_all.tsv')

molecules = spark.sql("SELECT * FROM kbase_msd_biochemistry.molecule").toPandas()
molecules.to_csv(f'{DATA_DIR}/molecules_all.tsv', sep='\t', index=False)
print(f'Cached {len(molecules):,} molecules to data/molecules_all.tsv')

reagents = spark.sql("SELECT * FROM kbase_msd_biochemistry.reagent").toPandas()
reagents.to_csv(f'{DATA_DIR}/reagents_all.tsv', sep='\t', index=False)
print(f'Cached {len(reagents):,} reagents to data/reagents_all.tsv')

## 11. Summary: Evidence Channel Viability

Based on all discoveries above, summarize which channels are viable and what the next steps are.

In [ ]:
print('='*60)
print('EVIDENCE CHANNEL VIABILITY SUMMARY')
print('='*60)
print()
print('Review outputs above and fill in this summary:')
print()
print('Channel 1 (KEGG via uniprot_identifier):  [check db values above]')
print('Channel 2 (BioCyc via uniprot_identifier): [check db values above]')
print('Channel 3 (EC numbers):                    [check all EC sources above]')
print('Channel 4 (PaperBLAST):                    [check curatedgene schema above]')
print('Channel 5 (InterPro):                      [check protein2ipr above]')
print('Channel 6 (SEED annotations):              [check seedannotation above]')
print('Channel 7 (Fuzzy name matching):            Always available (reaction.name)')
print()
print('Mass-balanced reactions:                    [check status values above]')
print('User DB extras:                            [check u_seaver diff above]')
print()
print('Next: Revise RESEARCH_PLAN.md based on these findings,')
print('      then build NB02-NB06 for confirmed channels.')